In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import math

def sample_8_percent_precise():
    """精确的8%抽样方法"""
    
    current_dir = os.getcwd()
    
    # 获取原始数据文件
    parquet_files = []
    for f in glob.glob(os.path.join(current_dir, "fhvhv_tripdata_2025-*.parquet")):
        parquet_files.append(f)
    
    if not parquet_files:
        print("没有找到2025年的行程数据文件")
        return
    
    print(f"找到 {len(parquet_files)} 个月份的数据文件")
    print("开始8%精确抽样处理...")
    print("=" * 60)
    
    all_samples = []
    monthly_stats = []
    total_original = 0
    total_sampled = 0
    
    for i, file_path in enumerate(parquet_files, 1):
        filename = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        month = filename.split('_')[-1].replace('.parquet', '')
        
        print(f"[{i}/{len(parquet_files)}] 处理 {month} 月:")
        print(f"   文件: {filename}")
        print(f"   大小: {file_size_mb:,.1f} MB")
        
        try:
            # 读取文件
            df = pd.read_parquet(file_path)
            original_rows = len(df)
            total_original += original_rows
            
            print(f"   总行数: {original_rows:,}")
            
            # 计算8%抽样数量
            target_sample = max(1, math.ceil(original_rows * 0.08))
            print(f"   目标抽样: {target_sample:,} 行 (8%)")
            
            if original_rows <= 12:  # 如果行数很少，全部保留
                sampled = df
                print(f"   → 行数较少，保留全部")
            else:
                # 使用高效抽样
                np.random.seed(2025 + i)
                
                if original_rows <= 5000000:  # 500万行以内
                    sampled = df.sample(frac=0.08, random_state=42, replace=False)
                else:
                    # 使用numpy抽样
                    sample_indices = np.random.choice(original_rows, size=target_sample, replace=False)
                    sampled = df.iloc[sample_indices].copy()
            
            sampled_rows = len(sampled)
            total_sampled += sampled_rows
            sample_percent = (sampled_rows / original_rows * 100) if original_rows > 0 else 100
            
            print(f"   ✅ 实际抽样: {sampled_rows:,} 行 ({sample_percent:.1f}%)")
            
            all_samples.append(sampled)
            
            monthly_stats.append({
                '月份': month,
                '原始行数': original_rows,
                '抽样行数': sampled_rows,
                '百分比': sample_percent
            })
            
            # 释放内存
            del df
            
        except MemoryError:
            print(f"   ⚠ 内存不足，使用系统抽样...")
            
            try:
                # 系统抽样：每12-13行取1行
                step = max(1, int(1 / 0.08))  # 约12.5
                
                # 读取前N行进行系统抽样
                read_rows = min(original_rows, 1000000)  # 最多读100万行
                df_sample = pd.read_parquet(file_path, nrows=read_rows)
                
                if len(df_sample) > 0:
                    # 系统抽样
                    sampled = df_sample.iloc[::step].copy()
                    
                    # 确保大约8%
                    if len(sampled) > read_rows * 0.10:  # 如果超过10%，调整
                        target = int(read_rows * 0.08)
                        sampled = sampled.head(target)
                    elif len(sampled) < read_rows * 0.06:  # 如果少于6%，调整
                        target = int(read_rows * 0.08)
                        additional = target - len(sampled)
                        if additional > 0:
                            remaining = df_sample.drop(sampled.index)
                            extra = remaining.sample(n=min(additional, len(remaining)), random_state=42)
                            sampled = pd.concat([sampled, extra])
                    
                    all_samples.append(sampled)
                    total_sampled += len(sampled)
                    
                    monthly_stats.append({
                        '月份': month,
                        '原始行数': f"{read_rows:,}(部分)",
                        '抽样行数': len(sampled),
                        '百分比': (len(sampled)/read_rows*100) if read_rows > 0 else 0
                    })
                    
                    print(f"   ✅ 系统抽样: {len(sampled):,} 行")
                
            except Exception as e:
                print(f"   ❌ 系统抽样失败: {e}")
                
        except Exception as e:
            print(f"   ❌ 处理失败: {e}")
    
    if all_samples:
        print(f"\n{'='*60}")
        print("合并所有抽样数据...")
        
        merged_df = pd.concat(all_samples, ignore_index=True)
        
        output_file = "2025_8percent_sampled.parquet"
        output_path = os.path.join(current_dir, output_file)
        
        print(f"保存到: {output_path}")
        print("正在保存，请稍候...")
        
        # 保存文件
        merged_df.to_parquet(output_path, index=False, compression='snappy')
        
        # 统计信息
        output_size_mb = os.path.getsize(output_path) / (1024 * 1024)
        estimated_original_size = sum(os.path.getsize(f) for f in parquet_files) / (1024 * 1024)
        
        print(f"\n{'='*60}")
        print("✅ 8%抽样完成!")
        print(f"{'='*60}")
        
        print(f"📊 总体统计:")
        print(f"  原始文件总数: {len(parquet_files)}")
        print(f"  原始总大小: {estimated_original_size:,.1f} MB")
        print(f"  原始总行数: {total_original:,}")
        print(f"  抽样总行数: {total_sampled:,}")
        
        if total_original > 0:
            overall_percent = (total_sampled / total_original * 100)
            print(f"  总体抽样比例: {overall_percent:.1f}%")
            size_reduction = ((estimated_original_size - output_size_mb) / estimated_original_size * 100)
            print(f"  大小缩减: {size_reduction:.1f}%")
            
            # 与25%对比
            estimated_25pct_size = estimated_original_size * 0.25
            reduction_vs_25pct = ((estimated_25pct_size - output_size_mb) / estimated_25pct_size * 100)
            print(f"  相比25%抽样减少: {reduction_vs_25pct:.1f}%")
        
        print(f"\n📁 输出文件:")
        print(f"  名称: {output_file}")
        print(f"  大小: {output_size_mb:,.1f} MB (~{output_size_mb/1024:.2f} GB)")
        print(f"  行数: {len(merged_df):,}")
        print(f"  列数: {len(merged_df.columns)}")
        
        # 月度详细统计
        print(f"\n📅 月度统计:")
        print("-" * 55)
        print(f"{'月份':<8} {'原始行数':>12} {'抽样行数':>12} {'比例':>8}")
        print("-" * 55)
        
        for stat in monthly_stats:
            if isinstance(stat['原始行数'], (int, float)):
                print(f"{stat['月份']:<8} {stat['原始行数']:>12,} {stat['抽样行数']:>12,} {stat['百分比']:>7.1f}%")
            else:
                print(f"{stat['月份']:<8} {stat['原始行数']:>12} {stat['抽样行数']:>12,} {'N/A':>8}")
        
        print("-" * 55)
        
        # 数据预览
        print(f"\n🔍 数据预览 (前5行):")
        print(merged_df.head())
        
        return merged_df
    
    return None

# 运行8%抽样
result = sample_8_percent_precise()



找到 12 个月份的数据文件
开始8%精确抽样处理...
[1/12] 处理 2025-01 月:
   文件: fhvhv_tripdata_2025-01.parquet
   大小: 468.3 MB
   总行数: 20,405,666
   目标抽样: 1,632,454 行 (8%)
   ✅ 实际抽样: 1,632,454 行 (8.0%)
[2/12] 处理 2025-02 月:
   文件: fhvhv_tripdata_2025-02.parquet
   大小: 440.2 MB
   总行数: 19,339,461
   目标抽样: 1,547,157 行 (8%)
   ✅ 实际抽样: 1,547,157 行 (8.0%)
[3/12] 处理 2025-03 月:
   文件: fhvhv_tripdata_2025-03.parquet
   大小: 478.5 MB
   总行数: 20,536,879
   目标抽样: 1,642,951 行 (8%)
   ✅ 实际抽样: 1,642,951 行 (8.0%)
[4/12] 处理 2025-04 月:
   文件: fhvhv_tripdata_2025-04.parquet
   大小: 464.6 MB
   总行数: 19,753,983
   目标抽样: 1,580,319 行 (8%)
   ✅ 实际抽样: 1,580,319 行 (8.0%)
[5/12] 处理 2025-05 月:
   文件: fhvhv_tripdata_2025-05.parquet
   大小: 493.4 MB
   总行数: 21,091,193
   目标抽样: 1,687,296 行 (8%)
   ✅ 实际抽样: 1,687,296 行 (8.0%)
[6/12] 处理 2025-06 月:
   文件: fhvhv_tripdata_2025-06.parquet
   大小: 468.1 MB
   总行数: 19,868,009
   目标抽样: 1,589,441 行 (8%)
   ✅ 实际抽样: 1,589,441 行 (8.0%)
[7/12] 处理 2025-07 月:
   文件: fhvhv_tripdata_2025-07.parquet
   大小: 464.7

In [ ]:
def sample_remaining_after_8pct(
    remaining_percent=0.02,
    seed_base=2025,
    output_suffix="_remaining_2pct.parquet"
):
    """
    在已抽取的 8% 之外，从剩余数据中再抽样
    """
    import pandas as pd
    import numpy as np
    import glob
    import os

    parquet_files = glob.glob("fhvhv_tripdata_2025-*.parquet")
    if not parquet_files:
        print("❌ 没有找到 parquet 文件")
        return

    all_remaining_samples = []

    for i, file_path in enumerate(parquet_files, 1):
        month = os.path.basename(file_path).split("_")[-1].replace(".parquet", "")
        print(f"[{i}] {month}")

        df = pd.read_parquet(file_path)
        n = len(df)

        # 复现第一次 8% 的索引
        np.random.seed(seed_base + i)
        first_n = max(1, int(n * 0.08))
        first_indices = np.random.choice(n, size=first_n, replace=False)

        # 剩余索引
        remaining_indices = np.setdiff1d(np.arange(n), first_indices)

        if len(remaining_indices) == 0:
            print("   ⚠ 无剩余数据")
            continue

        # 从剩余中抽样
        remaining_target = max(
            1,
            int(len(remaining_indices) * remaining_percent)
        )

        np.random.seed(seed_base + i + 5000)
        extra_indices = np.random.choice(
            remaining_indices,
            size=remaining_target,
            replace=False
        )

        sampled = df.iloc[extra_indices].copy()
        all_remaining_samples.append(sampled)

In [6]:
early_8pct = pd.read_parquet("2025_8percent_sampled.parquet")
early_8pct.columns.tolist()

['hvfhs_license_num',
 'dispatching_base_num',
 'originating_base_num',
 'request_datetime',
 'on_scene_datetime',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID',
 'trip_miles',
 'trip_time',
 'base_passenger_fare',
 'tolls',
 'bcf',
 'sales_tax',
 'congestion_surcharge',
 'airport_fee',
 'tips',
 'driver_pay',
 'shared_request_flag',
 'shared_match_flag',
 'access_a_ride_flag',
 'wav_request_flag',
 'wav_match_flag',
 'cbd_congestion_fee']

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# ==================================================
# ✅ 配置区（可按需修改）
# ==================================================
EARLY_8PCT_FILE = "2025_8percent_sampled.parquet"
OUTPUT_FILE = "2025_remaining_sample.parquet"

KEY_COLS = [
    "hvfhs_license_num",
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID"
]

SAMPLE_FRAC = 0.02     # 从剩余数据中抽 2%
RANDOM_SEED = 42
# ==================================================

# ==================================================
# 1️⃣ 读取早期 8% 抽样（只作为“黑名单”）
# ==================================================
early_8pct = pd.read_parquet(EARLY_8PCT_FILE)

# 早期 8% 的主键集合
early_keys = early_8pct[KEY_COLS].drop_duplicates()

# 列 & dtype 模板（保证格式完全一致）
COLUMNS_TEMPLATE = early_8pct.columns
DTYPE_TEMPLATE = early_8pct.dtypes

# ==================================================
# 2️⃣ 从原始 12 个月数据中抽“不重叠样本”
# ==================================================
new_samples = []

for f in sorted(glob.glob("fhvhv_tripdata_2025-*.parquet")):
    df = pd.read_parquet(f)

    # ✅ 去掉早期 8% 中已经存在的行程
    remaining = (
        df.merge(
            early_keys,
            on=KEY_COLS,
            how="left",
            indicator=True
        )
        .query("_merge == 'left_only'")
        .drop(columns="_merge")
    )

    if remaining.empty:
        continue

    # ✅ 抽样
    n = max(1, int(len(remaining) * SAMPLE_FRAC))
    sampled = remaining.sample(n=n, random_state=RANDOM_SEED)

    new_samples.append(sampled)

new_df = pd.concat(new_samples, ignore_index=True)

# ==================================================
# 3️⃣ 强制列格式与早期 8% 完全一致
# ==================================================
new_df = new_df[COLUMNS_TEMPLATE]
new_df = new_df.astype(DTYPE_TEMPLATE)

# ==================================================
# 4️⃣ 导出最终文件
# ==================================================
new_df.to_parquet(
    OUTPUT_FILE,
    index=False,
    compression="snappy"
)

print("✅ 剩余数据抽样完成")
print("输出文件:", OUTPUT_FILE)
print("行数:", len(new_df))
print("列数:", len(new_df.columns))